### Setup & Load Data

In [17]:
# =========================
# 0) Setup & Load Data
# =========================
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

FILE_PATH = "../data/gastat_foreign_trade_cleaned.xlsx"

df = pd.read_excel(FILE_PATH)
df.head(10)

,Country ID,Country,Year,Section ID,Section,Trade Flow ID,Trade Flow,Million SAR
0,abw,Aruba,2022,S03,"Animal and vegetable fats, oils, waxex and their products",1,Imports,0.000000
1,abw,Aruba,2022,S13,"Articles of stone, plaster, cement, ceramic, glass",1,Imports,0.011998
2,abw,Aruba,2022,S15,Base metals and their articles,1,Imports,0.198855
3,abw,Aruba,2022,S12,"Footwear, headgear, umbrellas, sticks",1,Imports,0.000000
4,abw,Aruba,2022,S01,Animals and their products,1,Imports,0.000000
5,abw,Aruba,2022,S16,"Machinery and mechanical appliances, electrical equipment, parts thereof",1,Imports,0.093459
6,abw,Aruba,2022,S18,"Optical, photographic, measuring, checking, medical instruments and apparatu...",1,Imports,0.047234
7,abw,Aruba,2022,S10,"Paper, paperboard and their articles",1,Imports,5.111555
8,abw,Aruba,2022,S14,"Precious stones or metals and their articles, jewelry",1,Imports,0.000000
9,abw,Aruba,2022,S07,"Plastics, rubber and their articles",1,Imports,1.074593


### Basic Shape, Columns, and Quick Snapshot

In [18]:
# =========================
# 1) Basic overview
# =========================
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
display(pd.DataFrame({"column": df.columns}))

print("\nRandom sample:")
df.sample(10, random_state=42)

Rows: 25107
Columns: 8

Column names:


,column
0,Country ID
1,Country
2,Year
3,Section ID
4,Section
5,Trade Flow ID
6,Trade Flow
7,Million SAR



Random sample:


,Country ID,Country,Year,Section ID,Section,Trade Flow ID,Trade Flow,Million SAR
4347,cmr,Cameroon,2023,S05,Mineral Products,2,Exports,7.971751
4059,chn,China,2024,S04,"Prepared foodstuffs, beverages and tobacco",2,Exports,6.902279
8654,gha,Ghana,2023,S10,"Paper, paperboard and their articles",1,Imports,0.000000
8510,geo,Georgia,2023,S11,Textiles and their articles,1,Imports,7.689239
4699,cog,Congo,2024,S18,"Optical, photographic, measuring, checking, medical instruments and apparatu...",2,Exports,0.000000
4420,cmr,Cameroon,2024,S17,"Vehicles, aircraft, vessels and associated transport equipment",1,Imports,0.000000
963,arg,Argentina,2024,S21,Works of arts and antiqes,1,Imports,0.000512
12545,lbn,Lebanon,2022,S19,Arms and Ammunition; Parts and Accessories Thereof,2,Exports,0.000000
8735,gha,Ghana,2024,S14,"Precious stones or metals and their articles, jewelry",2,Exports,0.000000
20044,sgp,Singapore,2023,S20,Miscellaneous Manufactured Articles,2,Exports,5.072698


### Data Types, Memory, and “What looks numeric but isn’t?”

In [19]:
# =========================
# 2) Types & memory
# =========================
display(df.dtypes.to_frame("dtype"))

# Deep memory usage (more accurate)
mem = df.memory_usage(deep=True).sort_values(ascending=False)
display(mem.to_frame("memory_bytes").head(20))

# Quick "object" columns that might be numeric
obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
display(pd.DataFrame({"object_cols": obj_cols}))

,dtype
Country ID,object
Country,object
Year,int64
Section ID,object
Section,object
Trade Flow ID,int64
Trade Flow,object
Million SAR,float64


,memory_bytes
Section,2332600
Country,1451139
Trade Flow,1405992
Section ID,1305564
Country ID,1305564
Trade Flow ID,200856
Year,200856
Million SAR,200856
Index,132


,object_cols
0,Country ID
1,Country
2,Section ID
3,Section
4,Trade Flow


### Missing Values (Counts + %)

In [20]:
# =========================
# 3) Missing values
# =========================
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

missing = (
    pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
    .query("missing_count > 0")
    .sort_values(["missing_count", "missing_pct"], ascending=False)
)

missing

,missing_count,missing_pct


### Duplicate Rows & Duplicate Keys

In [21]:
# =========================
# 4.1) Full-row duplicates
# =========================
dup_rows = df.duplicated().sum()
print("Duplicate full rows:", dup_rows)

if dup_rows > 0:
    display(df[df.duplicated()].head(20))

Duplicate full rows: 1381


,Country ID,Country,Year,Section ID,Section,Trade Flow ID,Trade Flow,Million SAR
78,abw,Aruba,2024,S02,Plant Products,1,Imports,0.0
196,afg,Afghanistan,2024,S03,"Animal and vegetable fats, oils, waxex and their products",2,Exports,0.0
206,afg,Afghanistan,2024,S08,"Skins, leather and their articles, handbags and similar",2,Exports,0.0
210,afg,Afghanistan,2024,S10,"Paper, paperboard and their articles",2,Exports,0.0
214,afg,Afghanistan,2024,S12,"Footwear, headgear, umbrellas, sticks",2,Exports,0.0
216,afg,Afghanistan,2024,S13,"Articles of stone, plaster, cement, ceramic, glass",2,Exports,0.0
218,afg,Afghanistan,2024,S14,"Precious stones or metals and their articles, jewelry",2,Exports,0.0
234,afg,Afghanistan,2024,S11,Textiles and their articles,2,Exports,0.0
236,afg,Afghanistan,2024,S17,"Vehicles, aircraft, vessels and associated transport equipment",2,Exports,0.0
240,afg,Afghanistan,2024,S09,"Wood, cork, plaiting materials and their articles",2,Exports,0.0


### Cardinality of Categorical Columns (How many unique values?)

In [22]:
# =========================
# 5) Categorical cardinality
# =========================
cat_cols = df.select_dtypes(include=["object", "category"]).columns

card = pd.DataFrame({
    "column": cat_cols,
    "n_unique": [df[c].nunique(dropna=True) for c in cat_cols],
    "top_value": [df[c].value_counts(dropna=True).head(1).index[0] if df[c].notna().any() else None for c in cat_cols],
    "top_count": [df[c].value_counts(dropna=True).head(1).iloc[0] if df[c].notna().any() else None for c in cat_cols],
}).sort_values("n_unique", ascending=False)

card

,column,n_unique,top_value,top_count
0,Country ID,199,afg,162
1,Country,199,Afghanistan,162
2,Section ID,21,S03,1236
3,Section,21,"Animal and vegetable fats, oils, waxex and their products",1236
4,Trade Flow,2,Imports,12585


### Numeric Summary (describe + quantiles)

In [24]:
# =========================
# 7) Numeric describe
# =========================
num_cols = df.select_dtypes(include=["number"]).columns

if len(num_cols) == 0:
    print("No numeric columns detected.")
else:
    display(df[num_cols].describe(percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]).T)

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
Year,25107.0,2023.209583,0.830232,2022.0,2022.0,2022.0,2022.0,2022.0,2023.00000,2024.000000,2024.000000,2024.000000,2024.000000,2024.000000
Trade Flow ID,25107.0,1.498745,0.500008,1.0,1.0,1.0,1.0,1.0,1.00000,2.000000,2.000000,2.000000,2.000000,2.000000
Million SAR,25107.0,125.135508,1026.970076,0.0,0.0,0.0,0.0,0.0,0.05863,5.740254,105.994256,387.538055,2563.303117,63230.179046


### Outlier Scan (IQR Method) — Numeric Columns Only

In [25]:
# =========================
# 8) IQR outlier scan
# =========================
if len(num_cols) > 0:
    outlier_report = []

    for c in num_cols:
        s = df[c].dropna()
        if s.empty:
            continue

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            outlier_report.append([c, len(s), 0, 0, 0.0, q1, q3, iqr])
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((s < lower) | (s > upper)).sum()
        outlier_report.append([c, len(s), outliers, lower, upper, round(outliers/len(s)*100, 2), q1, q3, iqr])

    out_df = pd.DataFrame(outlier_report, columns=["column","non_null","outlier_count","lower_bound","upper_bound","outlier_pct","q1","q3","iqr"])
    out_df.sort_values("outlier_count", ascending=False)
else:
    print("No numeric columns detected.")